# 🚀 ClaimGuard AI V3 (Elevenlabs TTS & Multi-Thread Safe)
Execute the cells below sequentially.

In [ ]:
# ==============================================================================
# CELL 1: Install Dependencies (FIXED & UPGRADED: Added Text-to-Speech deps)
# ==============================================================================

# Install Python packages (REMOVED faiss-gpu, added nest-asyncio)
!pip install -q pdf2image gradio faiss-cpu sentence-transformers requests reportlab nest-asyncio

# Install system dependencies
!apt-get update -y
!apt-get install -y poppler-utils tesseract-ocr

In [ ]:
# ================================
# CELL 2: 🔧 FIX OLLAMA INSTALL (KAGGLE)
# ================================

import os
import subprocess
import time
import requests

print("🧹 Cleaning old/broken Ollama install...")
!rm -rf /usr/local/lib/ollama
!rm -rf /usr/local/bin/ollama

print("📦 Installing required dependencies...")
!apt-get update -y
!apt-get install -y curl zstd ca-certificates

print("⬇️ Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("🔍 Verifying installation...")
!ls /usr/local/bin/ollama

# ================================
# 🚀 START OLLAMA SERVER
# ================================

print("🚀 Starting Ollama server...")

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle = 1 GPU
env["OLLAMA_NUM_PARALLEL"] = "4"
env["OLLAMA_MAX_LOADED_MODELS"] = "2"

ollama_process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(10)  # give time to start

# ================================
# ✅ CHECK IF RUNNING
# ================================

def check_ollama():
    try:
        r = requests.get("http://localhost:11434/api/tags")
        return r.status_code == 200
    except:
        return False

if check_ollama():
    print("✅ Ollama is RUNNING!")
else:
    print("❌ Ollama failed to start")
    print("Logs:")
    print(ollama_process.stderr.read().decode()[:500])

# ================================
# 📥 PULL MODEL
# ================================

print("📥 Pulling llama3 model (first time takes time)...")
!ollama pull llama3:8b

print("✅ Setup Complete!")

In [ ]:
# ==============================================================================
# CELL 3: Imports & Configuration (FIXED ASYNC LOOP)
# ==============================================================================

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import asyncio
import nest_asyncio

# ✅ FIX 1: Force single event loop usage globally to resolve Gradio crash
nest_asyncio.apply()

import io
import re
import json
import time
import base64
import requests
import faiss
import numpy as np
import torch
from pdf2image import convert_from_path
from sentence_transformers import SentenceTransformer
import gradio as gr

# API Keys & Configurations
OCR_SPACE_API_KEY = "K87093604688957"
OLLAMA_API_URL = "http://localhost:11434/api/generate"
ELEVENLABS_API_KEY = "sk_f8f0b0203444a36ae7cfa53baff0ae16b8042086b53d85c8"
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'

print("Loading Embedding Model (CPU safe mode)...")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cpu")
print("✅ Embedding model loaded successfully on CPU")

# Global stores
vector_db = None
clause_store = []
pdf_cache = {}

In [ ]:
# CELL 4: 1. OCR MODULE (OCR.space Integration)
def extract_text_from_pdf(pdf_path):
    print("Converting PDF to images for OCR...")
    images = convert_from_path(pdf_path, dpi=100) 
    full_text = ""
    
    for i, img in enumerate(images):
        print(f"Processing Page {i+1} via OCR.space API...")
        img_byte_arr = io.BytesIO()
        img.save(img_byte_arr, format='JPEG', quality=40) 
        base64_encoded = base64.b64encode(img_byte_arr.getvalue()).decode('utf-8')
        
        payload = {
            'apikey': OCR_SPACE_API_KEY,
            'base64Image': f"data:image/jpeg;base64,{base64_encoded}",
            'language': 'eng',
            'isOverlayRequired': False
        }
        
        try:
            response = requests.post('https://api.ocr.space/parse/image', data=payload)
            result = response.json()
            if result.get('IsErroredOnProcessing') == False:
                parsed_text = result['ParsedResults'][0]['ParsedText']
                full_text += parsed_text + "\n\n"
            else:
                print(f"API Error on Page {i+1}: {result.get('ErrorMessage')}")
        except Exception as e:
            print(f"Request failed for Page {i+1}: {e}")
            
        time.sleep(1) 
        
    return full_text

In [ ]:
# CELL 5: 2. NLP TEXT PROCESSING MODULE
def process_text(raw_text):
    print("Cleaning and chunking text...")
    cleaned = re.sub(r'\s+', ' ', raw_text).strip()
    words = cleaned.split()
    
    chunk_size = 300
    overlap = 50
    chunks = [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size - overlap)]

    categories = {
        "waiting_period": ["waiting", "days from inception", "delay"],
        "exclusion": ["excluded", "not covered", "exclusion", "except"],
        "co_payment": ["co-pay", "copayment", "co payment", "%"],
        "room_rent": ["room rent", "bed charges", "icu"],
        "sub_limit": ["sub-limit", "capped at", "maximum limit"],
        "network_rule": ["network", "empanelled", "hospital"]
    }
    
    structured_clauses = []
    
    for chunk in chunks:
        cls_tags = []
        lower_chunk = chunk.lower()
        for cat, keywords in categories.items():
            if any(k in lower_chunk for k in keywords):
                cls_tags.append(cat)
                
        structured_clauses.append({
            "text": chunk,
            "categories": cls_tags if cls_tags else ["general"]
        })
        
    print(f"Extracted {len(structured_clauses)} structural clause chunks.")
    return structured_clauses

In [ ]:
# CELL 6: 3. EMBEDDING + VECTOR DB (FAISS)
def build_vector_db(structured_clauses):
    global vector_db, clause_store
    clause_store = structured_clauses
    texts = [c['text'] for c in structured_clauses]
    
    print("Generating FAISS Embeddings...")
    embeddings = embedder.encode(texts).astype('float32')
    
    dimension = embeddings.shape[1]
    vector_db = faiss.IndexFlatL2(dimension)
    vector_db.add(embeddings)
    print(f"Vector DB initialized with {vector_db.ntotal} clause embeddings.")

def get_relevant_clauses(query, top_k=5):
    if not vector_db: return []
    query_vector = embedder.encode([query]).astype('float32')
    distances, indices = vector_db.search(query_vector, top_k)
    return [clause_store[idx] for idx in indices[0] if idx < len(clause_store)]

In [ ]:
# CELL 7: 4. SCENARIO PARSER (Ollama JSON Parsing)
def call_ollama(prompt, system_prompt, json_format=True):
    payload = {
        "model": "llama3:8b", # Matched model tag with CELL 2
        "prompt": prompt,
        "system": system_prompt,
        "stream": False
    }
    if json_format:
        payload["format"] = "json"
        
    try:
        response = requests.post(OLLAMA_API_URL, json=payload, timeout=120)
        response.raise_for_status()
        return response.json().get('response', '{}' if json_format else '')
    except Exception as e:
        print(f"Ollama inference error: {e}")
        return "{}" if json_format else ""

def parse_scenario(user_query):
    print("Parsing scenario intent via Llama 3...")
    sys_prompt = "Convert user healthcare scenario to structured JSON with exactly keys: treatment, hospital, urgency, input_raw, estimated_cost."
    response = call_ollama(user_query, system_prompt=sys_prompt)
    
    try:
        return json.loads(response)
    except:
        return {"input_raw": user_query, "hospital": "Unknown", "treatment": "Unknown"}

In [ ]:
# CELL 8: ELITE AI DECISION ENGINE
def elite_decision_engine(scenario_json, context_clauses):
    print("🧠 Engaging Elite AI Risk Analyst Engine...")
    context_text = "\n".join([f"- {c['text']}" for c in context_clauses])
    
    sys_prompt = """You are an Elite AI Risk Analyst, Financial Engineer, and Insurance Claim Auditor.
Your task is to act as an INTELLIGENT DECISION ENGINE. 
DO NOT just answer the query. You MUST:
1. Infer missing data
2. Perform calculations
3. Show step-by-step reasoning
4. Justify every output
5. Reveal hidden risks
6. Add insights that are NOT explicitly asked

For missing data, intelligently estimate (e.g., average treatment costs, ICU stays, medications). State "Assumption Made:" and "Reason:".
Act like an insurance company trying to reject the claim, exposing traps and loopholes.
Predict future policy states and provide optimization strategies for the user.

OUTPUT STRICTLY IN THIS EXACT VALID JSON FORMAT (No markdown, no preamble):
{
  "assumptions": ["Assumption Made: [..] Reason: [..]"],
  "calculation_steps": [
    "Step 1: Estimated total bill...",
    "Step 2: Applied constraints...",
    "Step 3: Deductions...",
    "Step 4: Final payout..."
  ],
  "final_payout": "string",
  "deductions": ["List of specific deductions with clause references"],
  "risks": ["Top 5 rejection risks / technical loopholes"],
  "hidden_insights": ["Insights NOT explicitly asked"],
  "claim_probability": "string",
  "risk_score": "string",
  "coverage_efficiency": "string",
  "future_prediction": "string",
  "optimization_suggestions": ["Actionable alternative strategies"]
}"""

    prompt = f"""
INPUT RECEIVED:
1. User Scenario & Partial Financials:
{json.dumps(scenario_json, indent=2)}

2. Policy Clauses (FAISS Retrieved Context):
{context_text}

Execute the Elite Decision Engine protocol now. Return ONLY valid JSON.
"""
    
    try:
        resp = call_ollama(prompt, system_prompt=sys_prompt, json_format=True)
        return json.loads(resp)
    except Exception as e:
        print(f"⚠️ Engine Execution Failed: {e}")
        return {}

def format_elite_report(audit_json):
    try:
        report = f"""
# 🧠 ELITE AI RISK ANALYST REPORT

### 📊 DECISION METRICS
- **Claim Approval Probability:** `{audit_json.get('claim_probability', 'N/A')}`
- **Financial Risk Score:** `{audit_json.get('risk_score', 'N/A')}`
- **Coverage Efficiency:** `{audit_json.get('coverage_efficiency', 'N/A')}`

### 🧩 INFERRED DATA & ASSUMPTIONS
{chr(10).join([f"- {a}" for a in audit_json.get('assumptions', [])])}

### 💰 FINANCIAL CALCULATIONS & PAYOUT
**Final Estimated Payout:** **{audit_json.get('final_payout', '₹0')}**

**Step-by-Step Reasoning:**
{chr(10).join([f"- {step}" for step in audit_json.get('calculation_steps', [])])}

**Applied Deductions:**
{chr(10).join([f"- {d}" for d in audit_json.get('deductions', [])])}

### ⚠️ ADVERSARIAL RISK ANALYSIS & LOOPHOLES
{chr(10).join([f"🚨 **RISK:** {r}" for r in audit_json.get('risks', [])])}

### 🔍 HIDDEN INSIGHTS
{chr(10).join([f"👀 {h}" for h in audit_json.get('hidden_insights', [])])}

### 🧬 FUTURE PREDICTION
{audit_json.get('future_prediction', 'N/A')}

### 🔄 STRATEGIC OPTIMIZATION
{chr(10).join([f"💡 {opt}" for opt in audit_json.get('optimization_suggestions', [])])}
"""
        return report
    except Exception as e:
        return f"Formatting error: {str(e)}"

In [ ]:
# CELL 9: 🔊 TEXT TO SPEECH (BENGALI OUTPUT API)
def generate_bengali_audio(english_report):
    print("🎙️ Translating Summary to Bengali (via Llama 3)...")
    sys_prompt = "You are an elite bilingual healthcare translator. Output STRICTLY the Bengali text without ANY English, markdown formatting, or greetings."
    summary_prompt = f"Summarize the following insurance report into 2 to 3 precise sentences regarding the final payout and biggest risks, and translate it ONLY into Bengali:\n\n{english_report}"
    
    bengali_translation = call_ollama(summary_prompt, system_prompt=sys_prompt, json_format=False).strip()
    
    if not bengali_translation:
        bengali_translation = "অনুগ্রহ করে আপনার পলিসি চেক করুন এবং আরও যাচাইকরণের জন্য অপেক্ষা করুন।" # Fallback baseline text
        
    print(f"Bengali Translation output: {bengali_translation[:100]}...")
    
    print("🔊 Generating speech via ElevenLabs API...")
    url = "https://api.elevenlabs.io/v1/text-to-speech/21m00Tcm4TlvDq8ikWAM" # Rachel Voice Model
    
    headers = {
        "Accept": "audio/mpeg",
        "Content-Type": "application/json",
        "xi-api-key": ELEVENLABS_API_KEY
    }
    
    data = {
        "text": bengali_translation,
        "model_id": "eleven_multilingual_v2",
        "voice_settings": {
            "stability": 0.5,
            "similarity_boost": 0.75
        }
    }
    
    try:
        response = requests.post(url, json=data, headers=headers)
        response.raise_for_status()
        
        audio_path = "bengali_report_summary.mp3"
        with open(audio_path, "wb") as f:
            f.write(response.content)
            
        print("✅ Bengali Audio generated successfully!")
        return audio_path
    except Exception as e:
        print(f"❌ ElevenLabs TTS Error: {e}")
        return None

In [ ]:
# CELL 10: FULL PIPELINE INTEGRATION
def run_claim_analysis(pdf_file, user_query):
    # 1. Input Processing & Caching logic (Sync, non-async)
    pdf_path = pdf_file.name if hasattr(pdf_file, 'name') else str(pdf_file)
    
    if pdf_path not in pdf_cache:
        raw_text = extract_text_from_pdf(pdf_path)
        clauses = process_text(raw_text)
        build_vector_db(clauses)
        pdf_cache[pdf_path] = True
    else:
        print("=> Using cached Embeddings for this PDF")

    # 2. Pipeline Execution
    scenario = parse_scenario(user_query)
    context = get_relevant_clauses(user_query, top_k=5)
    audit = elite_decision_engine(scenario, context)
    
    # 3. Formatted Markdown Display
    report = format_elite_report(audit)

    # 4. Generate Bengali Audio
    audio_path = generate_bengali_audio(report)
    
    return report, audio_path

In [ ]:
# CELL 11: AUTOMATED TEST CASE & UI
def create_dummy_pdf(filename="sample_policy.pdf"):
    print("Generating simulated insurance policy PDF...")
    try:
        from reportlab.pdfgen import canvas
        c = canvas.Canvas(filename)
        c.drawString(100, 750, "ClaimGuard Sample Health Insurance Policy")
        c.drawString(100, 730, "1. This policy covers all emergency treatments and accident surgeries.")
        c.drawString(100, 710, "2. A 10% co-payment (co-pay) is mandatory for all surgical claims.")
        c.drawString(100, 690, "3. Sub-limit: Room rent is capped at ₹5,000 per day.")
        c.drawString(100, 670, "4. Waiting period: 30 days initial waiting period from inception.")
        c.drawString(100, 650, "5. Exclusions: Cosmetic surgery is strictly excluded.")
        c.drawString(100, 630, "6. Empanelled network hospital Apollo is fully covered.")
        c.save()
    except Exception as e:
        print("Failed to build sample PDF:", e)

create_dummy_pdf()
print("\n" + "="*50)
print("🧪 RUNNING AUTOMATED TEST CASE")
print("="*50)
test_report, test_audio = run_claim_analysis("sample_policy.pdf", "Accident surgery tomorrow costing 50000 at Apollo")
print("\n" + test_report)
print(f"Audio Saved explicitly at: {test_audio}")
print("="*50 + "\n")

# ✅ FIX 4: NO ASYNC HANDLERS. Fully sync file wrapper bypassing concurrency.
def gradio_wrapper(pdf, query):
    if not pdf or not query: return "⚠️ Please upload a PDF and specify a scenario.", None
    try:
        return run_claim_analysis(pdf, query)
    except Exception as e:
        return f"Pipeline Error: {str(e)}", None

interface = gr.Interface(
    fn=gradio_wrapper,
    inputs=[
        gr.File(label="📄 Upload Insurance Policy (PDF)"),
        gr.Textbox(label="🏥 Enter Medical Claim Scenario", placeholder="e.g., Accident surgery tomorrow costing 50000...")
    ],
    outputs=[
        gr.Markdown(label="🛡️ Elite Risk Analyst Report"),
        gr.Audio(label="🔊 Play Bengali Audio Report")
    ],
    title="ClaimGuard AI 🤖 - Intelligent Decision Engine",
    description="Upload a policy document and describe a claim scenario to predict claim approval, run adversarial risk analysis, and uncover hidden pitfalls.",
    theme="huggingface"
)

# ✅ FIX 2 & 3: Ensure queue engine forces synchronization across single loop 
interface.queue().launch(share=True, debug=True)